# E7S — finite-shot truncation: does the rank hierarchy survive sampling?

E7 established the truncation structure on exact probabilities: triangles
need roughly the top 25 coordinates, C4 roughly 100, C5 roughly 200 to
400, C6 roughly 400 to 1000. Those are exact top-k probabilities; a real
sampler must discover and estimate them. E7S answers the review's most
important practical question by simulating finite-shot readout and
measuring what survives. The review's ladder S in {1024, 4096, 16384,
65536} is extended upward to {2^18, 2^20, 2^22, 2^24} because a
pre-registered pilot on the n = 14 census showed the review's ladder sits
entirely below the emergence region: C3 at k = 400 measures R² = −0.003
at 2^16 shots, +0.714 at 2^20, +0.942 at 2^22, and +0.995 at 2^24. The
mechanism is Section 15 of the framework: at β = 0 every cubic graph
produces the identical distribution, all graph discrimination enters at
mixer order, and the measured cross-graph variation of head coordinates
(~2e-7 per coordinate against head probabilities near 4e-2, census
geometry ~1e-4 in total L1) sits beneath multinomial noise until the
budget reaches roughly 2^18. The extended grid therefore contains the
turn-on of every hierarchy rung instead of a uniform floor.

**Sampling method.** S multinomial draws from the stored sorted
exact_vector treated as a categorical distribution; the empirical sorted
representation is the descending-sorted counts divided by S. Sampling
from the sorted vector is distributionally identical to sampling from the
unsorted Born distribution and sorting the empirical histogram, because
multinomial counts depend only on the probability multiset and the sort
discards labels. The statevector being sampled is the census pipeline's
stored output; the only new machinery is numpy's multinomial generator.

**Four measures, per the review.**
1. Decodability grid: ridge R² (verbatim protocol, seed-0 splits) on
   empirical sorted frequencies truncated to k in {25, 100, 400, 1000},
   for C3 through C6, at every budget. The S = infinity row is computed
   in-notebook on the exact sorted vectors at the same depths with the
   same protocol, so the reference is protocol-identical rather than
   quoted.
2. Discovery: the fraction of the exact top-k coordinates receiving at
   least one shot.
3. Ranking stability: Spearman correlation between empirical counts and
   exact probabilities over the exact top-k.
4. Cospectral separation against sampling noise: per pair, empirical L1
   between mates divided by the L1 between two independent samples of the
   same graph. A ratio near 1.0 means the pair is invisible at that
   budget.

**Pre-registrations.** The hierarchy is expected to emerge with budget in
the order it appears in the exact geometry: C3 first (measured turn-on
between 2^18 and 2^24 in the pilot), then C4, C5, and C6 at successively
larger budgets, with the S = infinity row as the ceiling every budget
approaches from below, so the E7 depth ladder acquires a shot-ladder
counterpart. The review's original budgets are expected to read at the
constant-predictor floor on every target, and that floor is a finding: the
exact-probability structure is certified real and is beneath sampling
noise at practical budgets on these censuses. The separation cell expects ratio near 1.0
for all 46 pairs at every budget, which converts the 10^10-to-10^14
shot-wall doctrine from an estimate into a measurement; any pair visibly
above 1.0 at 65536 shots would be a finding requiring investigation, not
celebration. Replicate variance is quantified at one representative cell
(S = 4096, k = 400, C5, five replicates) rather than across the whole
grid.

**Runtime.** Sampling is minutes and its cost is budget-independent. The
finite-shot ridge grid at n = 16 is the long pole, roughly two to four
hours over the eight-budget ladder; n = 14 is minutes. RUN_NS is the
session knob; results checkpoint per budget.

## Environment

In [1]:
import pickle
import time

from collections import defaultdict
from itertools import combinations

import numpy as np
import networkx as nx

from scipy import stats

from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

from tqdm.notebook import tqdm


from scipy.linalg import LinAlgWarning
import warnings
warnings.filterwarnings('ignore', category=LinAlgWarning)

In [2]:
SEED = 0
RIDGE_ALPHAS = np.logspace(-14, 2, 17)
NUM_FOLDS = 5

CENSUS_BASES = {
    14: "/kaggle/input/notebooks/lukemiller1987/n14-aaai-dataset/",
    16: "/kaggle/input/notebooks/lukemiller1987/n16-aaai-dataset/",
}
EXPECTED_CUBIC_COUNTS = {14: 509, 16: 4060}
EXPECTED_COSPECTRAL = {14: {'groups': 3, 'members': 6},
                       16: {'groups': 41, 'members': 83}}

SHOT_BUDGETS = (1024, 4096, 16384, 65536,
                262144, 1048576, 4194304, 16777216)   # 2^10..2^16, 2^18..2^24
TRUNCATION_DEPTHS = (25, 100, 400, 1000)
GRID_TARGETS = ('C3', 'C4', 'C5', 'C6')
NUM_CHEAP_REPLICATES = 3          # discovery / ranking / separation measures
VARIANCE_CELL = {'shots': 4096, 'depth': 400, 'target': 'C5',
                 'replicates': 5}  # sampling-variance quantification

# Session knob: which censuses this session runs.
RUN_NS = (14, 16)

## Load the censuses, verify vectors, compute identity-verified targets

The stored sorted vectors are asserted sorted and normalized; targets C3
through C6 are recomputed by the sanctioned method and the two cubic
trace identities are asserted on every graph.

In [3]:
CENSUS = {}
for num_vertices in RUN_NS:
    with open(CENSUS_BASES[num_vertices]
              + f"n{num_vertices}_data_dict.pkl", 'rb') as census_file:
        census_data = pickle.load(census_file)
    graph_keys = sorted(census_data)
    assert len(graph_keys) == EXPECTED_CUBIC_COUNTS[num_vertices]

    adjacency_matrices = np.stack(
        [census_data[graph_key]['adj_mat'] for graph_key in graph_keys]
    ).astype(np.int64)
    exact_vectors = np.vstack(
        [census_data[graph_key]['exact_vector'] for graph_key in graph_keys])
    assert exact_vectors.shape == (len(graph_keys), 1 << num_vertices)
    assert np.all(np.diff(exact_vectors, axis=1) <= 0)
    assert np.abs(exact_vectors.sum(axis=1) - 1.0).max() < 1e-12

    cycle_count_arrays = {name: np.zeros(len(graph_keys))
                          for name in GRID_TARGETS}
    for graph_position, adjacency_matrix in enumerate(
            tqdm(adjacency_matrices, desc=f'n={num_vertices} targets')):
        cycle_counter = defaultdict(int)
        graph = nx.from_numpy_array(adjacency_matrix)
        for cycle_vertices in nx.simple_cycles(graph, length_bound=6):
            cycle_counter[len(cycle_vertices)] += 1
        for cycle_length, target_name in ((3, 'C3'), (4, 'C4'),
                                          (5, 'C5'), (6, 'C6')):
            cycle_count_arrays[target_name][graph_position] = (
                cycle_counter[cycle_length])

    cubed_traces = np.trace(adjacency_matrices @ adjacency_matrices
                            @ adjacency_matrices, axis1=1, axis2=2)
    fourth_traces = np.trace(np.stack(
        [np.linalg.matrix_power(adjacency_matrix, 4)
         for adjacency_matrix in adjacency_matrices]), axis1=1, axis2=2)
    assert np.array_equal(cubed_traces,
                          6 * cycle_count_arrays['C3'].astype(np.int64))
    assert np.array_equal(fourth_traces,
                          8 * cycle_count_arrays['C4'].astype(np.int64)
                          + 15 * num_vertices)

    CENSUS[num_vertices] = {'adjacency_matrices': adjacency_matrices,
                            'exact_vectors': exact_vectors,
                            'targets': cycle_count_arrays}
    print(f'n={num_vertices}: {len(graph_keys)} graphs, vectors verified, '
          f'both trace identities exact')

n=14 targets:   0%|          | 0/509 [00:00<?, ?it/s]

n=14: 509 graphs, vectors verified, both trace identities exact


n=16 targets:   0%|          | 0/4060 [00:00<?, ?it/s]

n=16: 4060 graphs, vectors verified, both trace identities exact


## Signal-versus-noise arithmetic

The one table that makes every other table interpretable: the cross-graph
standard deviation of the exact head coordinates (the signal a probe must
read) against the per-coordinate multinomial noise sqrt(p/S) at every
budget. The ridge aggregates across coordinates and therefore recovers
signal before the per-coordinate ratio reaches one, but the column
gradient predicts the ordering of the grid's turn-on points.

In [4]:
HEAD_DEPTH_FOR_DIAGNOSTIC = 25
for num_vertices in RUN_NS:
    exact_vectors = CENSUS[num_vertices]['exact_vectors']
    cross_graph_sd = exact_vectors[:, :HEAD_DEPTH_FOR_DIAGNOSTIC].std(axis=0)
    mean_head_probability = float(
        exact_vectors[:, :HEAD_DEPTH_FOR_DIAGNOSTIC].mean())
    print(f'n={num_vertices}: cross-graph sd of top-{HEAD_DEPTH_FOR_DIAGNOSTIC} '
          f'coords median {np.median(cross_graph_sd):.2e} '
          f'(max {cross_graph_sd.max():.2e}); '
          f'mean head probability {mean_head_probability:.2e}')
    for shot_budget in SHOT_BUDGETS:
        per_coordinate_noise = float(
            np.sqrt(mean_head_probability / shot_budget))
        print(f'  S={shot_budget:>9}: per-coordinate noise '
              f'{per_coordinate_noise:.2e}  '
              f'(median signal / noise '
              f'{float(np.median(cross_graph_sd)) / per_coordinate_noise:.3f})')

n=14: cross-graph sd of top-25 coords median 2.16e-07 (max 2.22e-06); mean head probability 3.96e-02
  S=     1024: per-coordinate noise 6.22e-03  (median signal / noise 0.000)
  S=     4096: per-coordinate noise 3.11e-03  (median signal / noise 0.000)
  S=    16384: per-coordinate noise 1.55e-03  (median signal / noise 0.000)
  S=    65536: per-coordinate noise 7.77e-04  (median signal / noise 0.000)
  S=   262144: per-coordinate noise 3.89e-04  (median signal / noise 0.001)
  S=  1048576: per-coordinate noise 1.94e-04  (median signal / noise 0.001)
  S=  4194304: per-coordinate noise 9.72e-05  (median signal / noise 0.002)
  S= 16777216: per-coordinate noise 4.86e-05  (median signal / noise 0.004)
n=16: cross-graph sd of top-25 coords median 2.13e-07 (max 2.12e-06); mean head probability 3.94e-02
  S=     1024: per-coordinate noise 6.21e-03  (median signal / noise 0.000)
  S=     4096: per-coordinate noise 3.10e-03  (median signal / noise 0.000)
  S=    16384: per-coordinate noise 1.

## Sampling machinery

One generator per (census, budget, replicate), seeded from (SEED, n, S,
replicate), draws every graph in index order, so every sample in the
notebook is reproducible from the config. The stored vectors carry a
~1e-15 normalization residue from the producer's float64 pipeline;
multinomial sampling requires an exact simplex point, so each vector is
renormalized by its own sum before sampling, a rescaling eleven orders of
magnitude below the smallest quantity this experiment measures.

In [5]:
def sample_shot_counts(exact_vectors, shot_budget, replicate_index,
                       num_vertices):
    """Multinomial shot counts for every graph, aligned to the exact
    sorted coordinates. Returns an int64 matrix of shape (graphs, 2^n)."""
    generator = np.random.default_rng(
        [SEED, num_vertices, shot_budget, replicate_index])
    count_matrix = np.empty(exact_vectors.shape, dtype=np.int64)
    for graph_position in range(exact_vectors.shape[0]):
        probabilities = exact_vectors[graph_position]
        probabilities = probabilities / probabilities.sum()
        count_matrix[graph_position] = generator.multinomial(shot_budget,
                                                             probabilities)
    return count_matrix


def empirical_sorted_features(count_matrix, shot_budget, feature_depth):
    """Descending-sorted empirical frequencies, truncated to feature_depth.
    Sorting is over the counts themselves; the alignment to exact
    coordinates is deliberately discarded, exactly as a real sampler's
    sorted readout would discard it."""
    sorted_counts = -np.sort(-count_matrix, axis=1)[:, :feature_depth]
    return sorted_counts.astype(np.float64) / shot_budget


def ridge_grid_cell(feature_matrix, target_values, fold_splits):
    """Verbatim probe: per-fold RidgeCV, R^2 on test. Returns fold scores."""
    fold_scores = []
    for train_indices, test_indices in fold_splits:
        probe_model = RidgeCV(alphas=RIDGE_ALPHAS, cv=5)
        probe_model.fit(feature_matrix[train_indices],
                        target_values[train_indices])
        fold_scores.append(r2_score(
            target_values[test_indices],
            probe_model.predict(feature_matrix[test_indices])))
    return np.array(fold_scores)

## The S = infinity reference row

The exact sorted vectors probed at the same depths with the same
protocol. Every finite budget approaches these values from below; the
row is computed rather than quoted so the comparison is
protocol-identical.

In [6]:
EXACT_ROW = {}
for num_vertices in RUN_NS:
    exact_vectors = CENSUS[num_vertices]['exact_vectors']
    fold_generator = KFold(n_splits=NUM_FOLDS, shuffle=True,
                           random_state=SEED)
    fold_splits = list(fold_generator.split(np.arange(len(exact_vectors))))
    CENSUS[num_vertices]['fold_splits'] = fold_splits

    EXACT_ROW[num_vertices] = {}
    for feature_depth in TRUNCATION_DEPTHS:
        exact_features = exact_vectors[:, :feature_depth]
        for target_name in GRID_TARGETS:
            fold_scores = ridge_grid_cell(
                exact_features, CENSUS[num_vertices]['targets'][target_name],
                fold_splits)
            EXACT_ROW[num_vertices][(feature_depth, target_name)] = {
                'r2_mean': float(fold_scores.mean()),
                'r2_sd': float(fold_scores.std())}
        depth_line = '  '.join(
            f"{target_name} "
            f"{EXACT_ROW[num_vertices][(feature_depth, target_name)]['r2_mean']:+.3f}"
            for target_name in GRID_TARGETS)
        print(f'n={num_vertices} S=inf k={feature_depth:>5}: {depth_line}')

n=14 S=inf k=   25: C3 +1.000  C4 +0.957  C5 +0.034  C6 +0.163
n=14 S=inf k=  100: C3 +1.000  C4 +0.993  C5 +0.272  C6 +0.235
n=14 S=inf k=  400: C3 +1.000  C4 +0.998  C5 +0.927  C6 +0.478
n=14 S=inf k= 1000: C3 +1.000  C4 +0.998  C5 +0.928  C6 +0.484
n=16 S=inf k=   25: C3 +1.000  C4 +0.968  C5 +0.169  C6 +0.126
n=16 S=inf k=  100: C3 +1.000  C4 +0.996  C5 +0.335  C6 +0.184
n=16 S=inf k=  400: C3 +1.000  C4 +0.998  C5 +0.981  C6 +0.363
n=16 S=inf k= 1000: C3 +1.000  C4 +1.000  C5 +0.982  C6 +0.640


## Finite-shot decodability grid

One sampling replicate (replicate 0) per budget carries the grid;
sampling variance is quantified separately at the pre-registered cell.
Results checkpoint after every budget.

In [7]:
GRID_RESULTS = {}
for num_vertices in RUN_NS:
    exact_vectors = CENSUS[num_vertices]['exact_vectors']
    fold_splits = CENSUS[num_vertices]['fold_splits']
    GRID_RESULTS[num_vertices] = {}

    for shot_budget in SHOT_BUDGETS:
        count_matrix = sample_shot_counts(exact_vectors, shot_budget,
                                          replicate_index=0,
                                          num_vertices=num_vertices)
        for feature_depth in TRUNCATION_DEPTHS:
            feature_matrix = empirical_sorted_features(
                count_matrix, shot_budget, feature_depth)
            for target_name in tqdm(
                    GRID_TARGETS,
                    desc=f'n={num_vertices} S={shot_budget} k={feature_depth}'):
                fold_scores = ridge_grid_cell(
                    feature_matrix,
                    CENSUS[num_vertices]['targets'][target_name], fold_splits)
                GRID_RESULTS[num_vertices][
                    (shot_budget, feature_depth, target_name)] = {
                    'r2_folds': fold_scores,
                    'r2_mean': float(fold_scores.mean()),
                    'r2_sd': float(fold_scores.std())}
        with open('/kaggle/working/e7s_finite_shot.pkl', 'wb') as output_file:
            pickle.dump({'grid_results': GRID_RESULTS,
                         'exact_row': EXACT_ROW}, output_file)
        print(f'n={num_vertices} S={shot_budget}: checkpointed')

n=14 S=1024 k=25:   0%|          | 0/4 [00:00<?, ?it/s]

n=14 S=1024 k=100:   0%|          | 0/4 [00:00<?, ?it/s]

n=14 S=1024 k=400:   0%|          | 0/4 [00:00<?, ?it/s]

n=14 S=1024 k=1000:   0%|          | 0/4 [00:00<?, ?it/s]

n=14 S=1024: checkpointed


...

n=16 S=16777216 k=25:   0%|          | 0/4 [00:00<?, ?it/s]

n=16 S=16777216 k=100:   0%|          | 0/4 [00:00<?, ?it/s]

n=16 S=16777216 k=400:   0%|          | 0/4 [00:00<?, ?it/s]

n=16 S=16777216 k=1000:   0%|          | 0/4 [00:00<?, ?it/s]

n=16 S=16777216: checkpointed


## Sampling variance at the pre-registered cell

Five independent sampling replicates at (S = 4096, k = 400, C5): the
spread of the fold-mean R² across replicates is the sampling-noise scale
for the whole grid, quantified once where the hierarchy is most
sensitive.

In [8]:
VARIANCE_RESULTS = {}
for num_vertices in RUN_NS:
    replicate_means = []
    for replicate_index in range(VARIANCE_CELL['replicates']):
        count_matrix = sample_shot_counts(
            CENSUS[num_vertices]['exact_vectors'], VARIANCE_CELL['shots'],
            replicate_index, num_vertices)
        feature_matrix = empirical_sorted_features(
            count_matrix, VARIANCE_CELL['shots'], VARIANCE_CELL['depth'])
        fold_scores = ridge_grid_cell(
            feature_matrix,
            CENSUS[num_vertices]['targets'][VARIANCE_CELL['target']],
            CENSUS[num_vertices]['fold_splits'])
        replicate_means.append(float(fold_scores.mean()))
    VARIANCE_RESULTS[num_vertices] = replicate_means
    print(f"n={num_vertices} {VARIANCE_CELL['target']} at "
          f"S={VARIANCE_CELL['shots']} k={VARIANCE_CELL['depth']}: "
          f"replicate means {[f'{value:+.3f}' for value in replicate_means]} "
          f"(sd {np.std(replicate_means):.4f})")

n=14 C5 at S=4096 k=400: replicate means ['-0.013', '-0.013', '-0.013', '-0.021', '-0.019'] (sd 0.0036)
n=16 C5 at S=4096 k=400: replicate means ['-0.001', '-0.003', '-0.001', '-0.001', '-0.001'] (sd 0.0008)


## Discovery and ranking stability

Per budget, averaged over three replicates and all graphs: the fraction
of exact top-k coordinates receiving at least one shot, and the Spearman
correlation between shot counts and exact probabilities over the exact
top-k. These use the count alignment to exact coordinates, which the
sorted readout discards but the analysis may keep.

In [9]:
DISCOVERY_RESULTS = {}
for num_vertices in RUN_NS:
    exact_vectors = CENSUS[num_vertices]['exact_vectors']
    DISCOVERY_RESULTS[num_vertices] = {}
    for shot_budget in SHOT_BUDGETS:
        discovery_accumulator = {depth: [] for depth in TRUNCATION_DEPTHS}
        spearman_accumulator = {depth: [] for depth in TRUNCATION_DEPTHS}
        for replicate_index in range(NUM_CHEAP_REPLICATES):
            count_matrix = sample_shot_counts(exact_vectors, shot_budget,
                                              replicate_index, num_vertices)
            for depth in TRUNCATION_DEPTHS:
                head_counts = count_matrix[:, :depth]
                discovery_accumulator[depth].append(
                    (head_counts > 0).mean())
                spearman_values = [
                    stats.spearmanr(head_counts[graph_position],
                                    exact_vectors[graph_position, :depth]
                                    ).statistic
                    for graph_position in range(len(exact_vectors))]
                spearman_accumulator[depth].append(
                    float(np.nanmean(spearman_values)))
        DISCOVERY_RESULTS[num_vertices][shot_budget] = {
            depth: {'discovery': float(np.mean(discovery_accumulator[depth])),
                    'spearman': float(np.mean(spearman_accumulator[depth]))}
            for depth in TRUNCATION_DEPTHS}
        summary_line = '  '.join(
            f"k={depth}: disc "
            f"{DISCOVERY_RESULTS[num_vertices][shot_budget][depth]['discovery']:.3f} "
            f"rho {DISCOVERY_RESULTS[num_vertices][shot_budget][depth]['spearman']:.3f}"
            for depth in TRUNCATION_DEPTHS)
        print(f'n={num_vertices} S={shot_budget:>5}: {summary_line}')

n=14 S= 1024: k=25: disc 0.765 rho 0.761  k=100: disc 0.265 rho 0.661  k=400: disc 0.069 rho 0.410  k=1000: disc 0.028 rho 0.276
n=14 S= 4096: k=25: disc 0.954 rho 0.764  k=100: disc 0.440 rho 0.737  k=400: disc 0.123 rho 0.509  k=1000: disc 0.049 rho 0.357
n=14 S=16384: k=25: disc 1.000 rho 0.761  k=100: disc 0.686 rho 0.748  k=400: disc 0.219 rho 0.607  k=1000: disc 0.089 rho 0.457
n=14 S=65536: k=25: disc 1.000 rho 0.763  k=100: disc 0.963 rho 0.747  k=400: disc 0.395 rho 0.734  k=1000: disc 0.164 rho 0.589
n=14 S=262144: k=25: disc 1.000 rho 0.764  k=100: disc 1.000 rho 0.761  k=400: disc 0.637 rho 0.817  k=1000: disc 0.277 rho 0.710
n=14 S=1048576: k=25: disc 1.000 rho 0.763  k=100: disc 1.000 rho 0.792  k=400: disc 0.847 rho 0.893  k=1000: disc 0.421 rho 0.805
n=14 S=4194304: k=25: disc 1.000 rho 0.765  k=100: disc 1.000 rho 0.846  k=400: disc 0.951 rho 0.913  k=1000: disc 0.622 rho 0.853
n=14 S=16777216: k=25: disc 1.000 rho 0.767  k=100: disc 1.000 rho 0.903  k=400: disc 0.999 

## Cospectral separation against sampling noise

The exact integer census is recomputed and asserted against E8A. Per
pair, per budget: the L1 between the mates' empirical sorted vectors,
divided by the same-replicate L1 between two independent samples of the
same graph. The pre-registered expectation is ratio near 1.0 everywhere,
which measures the shot wall rather than estimating it.

In [10]:
SEPARATION_RESULTS = {}
for num_vertices in RUN_NS:
    adjacency_matrices = CENSUS[num_vertices]['adjacency_matrices']
    census_size = len(adjacency_matrices)
    trace_tuples = np.zeros((census_size, num_vertices), dtype=np.int64)
    walk_matrix = adjacency_matrices.copy()
    trace_tuples[:, 0] = np.trace(walk_matrix, axis1=1, axis2=2)
    for power_index in range(1, num_vertices):
        walk_matrix = walk_matrix @ adjacency_matrices
        trace_tuples[:, power_index] = np.trace(walk_matrix, axis1=1, axis2=2)
    groups_by_tuple = defaultdict(list)
    for graph_index in range(census_size):
        groups_by_tuple[tuple(trace_tuples[graph_index])].append(graph_index)
    cospectral_groups = sorted(sorted(member_list) for member_list
                               in groups_by_tuple.values()
                               if len(member_list) > 1)
    member_indices = sorted({graph_index for group in cospectral_groups
                             for graph_index in group})
    assert len(cospectral_groups) == EXPECTED_COSPECTRAL[num_vertices]['groups']
    assert len(member_indices) == EXPECTED_COSPECTRAL[num_vertices]['members']

    member_vectors = CENSUS[num_vertices]['exact_vectors'][member_indices]
    member_position = {graph_index: position for position, graph_index
                       in enumerate(member_indices)}

    SEPARATION_RESULTS[num_vertices] = {}
    for shot_budget in SHOT_BUDGETS:
        # two independent samples of every member per replicate: replicate
        # index r gives sample A, replicate index r + 100 gives sample B
        pair_ratio_accumulator = []
        for replicate_index in range(NUM_CHEAP_REPLICATES):
            sample_a = np.sort(sample_shot_counts(
                member_vectors, shot_budget, replicate_index,
                num_vertices))[:, ::-1] / shot_budget
            sample_b = np.sort(sample_shot_counts(
                member_vectors, shot_budget, replicate_index + 100,
                num_vertices))[:, ::-1] / shot_budget
            for group in cospectral_groups:
                for first_index, second_index in combinations(group, 2):
                    first_position = member_position[first_index]
                    second_position = member_position[second_index]
                    mate_l1 = np.abs(sample_a[first_position]
                                     - sample_a[second_position]).sum()
                    noise_l1 = 0.5 * (
                        np.abs(sample_a[first_position]
                               - sample_b[first_position]).sum()
                        + np.abs(sample_a[second_position]
                                 - sample_b[second_position]).sum())
                    pair_ratio_accumulator.append(mate_l1 / noise_l1)
        ratios = np.array(pair_ratio_accumulator)
        SEPARATION_RESULTS[num_vertices][shot_budget] = {
            'ratio_mean': float(ratios.mean()),
            'ratio_max': float(ratios.max()),
            'ratio_min': float(ratios.min())}
        print(f'n={num_vertices} S={shot_budget:>5}: mate-L1 / noise-L1 '
              f'mean {ratios.mean():.3f}, range '
              f'[{ratios.min():.3f}, {ratios.max():.3f}] '
              f'({len(ratios)} pair-replicates)')

n=14 S= 1024: mate-L1 / noise-L1 mean 1.054, range [0.500, 2.000] (9 pair-replicates)
n=14 S= 4096: mate-L1 / noise-L1 mean 1.177, range [0.492, 2.512] (9 pair-replicates)
n=14 S=16384: mate-L1 / noise-L1 mean 1.068, range [0.737, 1.720] (9 pair-replicates)
n=14 S=65536: mate-L1 / noise-L1 mean 1.060, range [0.476, 1.888] (9 pair-replicates)
n=14 S=262144: mate-L1 / noise-L1 mean 0.868, range [0.593, 1.213] (9 pair-replicates)
n=14 S=1048576: mate-L1 / noise-L1 mean 1.014, range [0.612, 1.253] (9 pair-replicates)
n=14 S=4194304: mate-L1 / noise-L1 mean 0.873, range [0.587, 1.692] (9 pair-replicates)
n=14 S=16777216: mate-L1 / noise-L1 mean 1.215, range [0.932, 1.654] (9 pair-replicates)
n=16 S= 1024: mate-L1 / noise-L1 mean 1.019, range [0.245, 2.889] (129 pair-replicates)
n=16 S= 4096: mate-L1 / noise-L1 mean 1.048, range [0.316, 2.429] (129 pair-replicates)
n=16 S=16384: mate-L1 / noise-L1 mean 1.080, range [0.353, 2.217] (129 pair-replicates)
n=16 S=65536: mate-L1 / noise-L1 mean 1.

## Summary tables and persistence

In [11]:
for num_vertices in RUN_NS:
    print(f'\n=== n={num_vertices} — decodability R^2(S, k) ===')
    for target_name in GRID_TARGETS:
        print(f'\n{target_name}:')
        header = '  '.join(f'k={depth:>5}' for depth in TRUNCATION_DEPTHS)
        print(f'{"S":>8} | {header}')
        for shot_budget in SHOT_BUDGETS:
            row_cells = '  '.join(
                f"{GRID_RESULTS[num_vertices][(shot_budget, depth, target_name)]['r2_mean']:+7.3f}"
                for depth in TRUNCATION_DEPTHS)
            print(f'{shot_budget:>8} | {row_cells}')
        exact_cells = '  '.join(
            f"{EXACT_ROW[num_vertices][(depth, target_name)]['r2_mean']:+7.3f}"
            for depth in TRUNCATION_DEPTHS)
        print(f'{"inf":>8} | {exact_cells}')

with open('/kaggle/working/e7s_finite_shot.pkl', 'wb') as output_file:
    pickle.dump({'grid_results': GRID_RESULTS,
                 'exact_row': EXACT_ROW,
                 'variance_cell': {'spec': VARIANCE_CELL,
                                   'replicate_means': VARIANCE_RESULTS},
                 'discovery_results': DISCOVERY_RESULTS,
                 'separation_results': SEPARATION_RESULTS,
                 'config': {'seed': SEED, 'shot_budgets': SHOT_BUDGETS,
                            'truncation_depths': TRUNCATION_DEPTHS,
                            'grid_targets': GRID_TARGETS,
                            'cheap_replicates': NUM_CHEAP_REPLICATES,
                            'run_ns': RUN_NS}}, output_file)
print('\nsaved e7s_finite_shot.pkl')


=== n=14 — decodability R^2(S, k) ===

C3:
       S | k=   25  k=  100  k=  400  k= 1000
    1024 |  -0.004   -0.004   -0.004   -0.004
    4096 |  -0.025   -0.024   -0.024   -0.024
   16384 |  -0.004   -0.004   -0.004   -0.004
   65536 |  -0.012   -0.003   -0.003   -0.003
  262144 |  -0.003   -0.003   -0.003   -0.003
 1048576 |  -0.011   +0.065   +0.714   +0.714
 4194304 |  -0.004   +0.175   +0.942   +0.942
16777216 |  +0.031   +0.598   +0.995   +0.995
     inf |  +1.000   +1.000   +1.000   +1.000

C4:
       S | k=   25  k=  100  k=  400  k= 1000
    1024 |  -0.008   -0.017   -0.017   -0.017
    4096 |  -0.011   -0.011   -0.011   -0.011
   16384 |  -0.001   -0.001   -0.001   -0.001
   65536 |  -0.010   -0.010   -0.010   -0.010
  262144 |  -0.008   -0.008   -0.008   -0.008
 1048576 |  -0.011   -0.025   -0.020   -0.020
 4194304 |  -0.009   +0.205   +0.225   +0.228
16777216 |  -0.004   +0.667   +0.688   +0.688
     inf |  +0.957   +0.993   +0.998   +0.998

C5:
       S | k=   25  k=  10

## Results

(Written after the run. The reading order: (1) the S = infinity row
against the E7 record as a protocol consistency check; (2) each target's
column read upward from S = 1024 — the review's original budgets are
expected at the floor, the emergence budget of each rung (the smallest S
whose cell reaches, say, half its S = infinity value) is the paper's
finite-shot sentence for that rung, and the pre-registered ordering is
C3 before C4 before C5 before C6, extending the E7 depth ladder into a
shot ladder;
(3) discovery and Spearman against the R² grid — decodability should
track head discovery and rank stability, and a cell where R² survives
despite poor discovery would be a finding; (4) the separation ratios,
expected near 1.0 everywhere: that table is the measured shot wall, and
it is reported as the closing fact of the no-advantage discipline;
(5) the variance cell, which bounds how much of any grid cell's
fluctuation is sampling noise. Verb discipline: budgets "preserve" or
"do not preserve" linear decodability at a depth; nothing here concerns
hardware, noise models, or efficiency claims.)